# ResNet-18 convolutional baseline -- Colab runner

The comparison requested by the supervisor: **ResNet-18 with plain cross-entropy
versus ResNet-18 with cross-entropy + Center Loss**, on the same room dataset, so
the convolutional baseline can be set against the ViT-S/16 results.

Every training condition is identical to the ViT runs -- same split, optimizer,
LR schedule, epochs, lambda and seed -- so the **architecture is the only thing
that differs**.

Run the cells top to bottom on a **T4 GPU** runtime. Cell 5 does everything:
both variants, both evaluations, and the summary table.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Get the code and install dependencies

Clones the repository and installs the shared requirements. `resnet18_baseline/`
borrows its training loop from `vit_s16_baseline/`, so both must be present --
cloning the repo gives you the second one, and cell 3b makes sure of the first.

In [ ]:
import os

REPO_DIR = '/content/room-classification'
BRANCH   = 'main'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/SilviuBR24/room-classification.git {REPO_DIR}
!cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

# Install from the ViT project: it holds the shared requirements, and it is
# always present in the clone. Absolute path on purpose -- resnet18_baseline
# may not be in the repository yet, so we must not cd into it here. Cell 3b
# puts it in place (from Drive if needed) and cell 5 changes into it.
!pip install -q -r {REPO_DIR}/vit_s16_baseline/requirements.txt

%cd {REPO_DIR}
print('Code ready on branch:', BRANCH)
print('resnet18_baseline present in clone:',
      os.path.isdir(f'{REPO_DIR}/resnet18_baseline'))

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3b. Make sure the ResNet code is present

`resnet18_baseline/` may not be on GitHub yet. If the clone does not contain it,
this copies the folder from Drive instead, so the run works either way. Once the
code is pushed to `main`, this cell simply reports that nothing was needed.

In [ ]:
import os, shutil

REPO_FOLDER  = '/content/room-classification/resnet18_baseline'
DRIVE_FOLDER = '/content/drive/MyDrive/Dissertation_Thesis/resnet18_baseline'

needed = ['compare_resnet18.py', 'train_resnet18.py', 'evaluate_resnet18.py',
          'resnet_model.py', 'shared_infrastructure.py', 'config_resnet18.yaml']

missing = [f for f in needed if not os.path.isfile(os.path.join(REPO_FOLDER, f))]

if not missing:
    print('resnet18_baseline already present in the clone -- nothing to do.')
else:
    assert os.path.isdir(DRIVE_FOLDER), (
        f'{REPO_FOLDER} is incomplete and {DRIVE_FOLDER} does not exist. '
        f'Either push the code to main, or upload resnet18_baseline/ to Drive.')
    os.makedirs(REPO_FOLDER, exist_ok=True)
    for f in os.listdir(DRIVE_FOLDER):
        src = os.path.join(DRIVE_FOLDER, f)
        if os.path.isfile(src):
            shutil.copy(src, os.path.join(REPO_FOLDER, f))
    print(f'Copied resnet18_baseline from Drive (was missing: {missing})')

# The ResNet folder borrows its training loop from the ViT project next to it.
assert os.path.isdir('/content/room-classification/vit_s16_baseline'),     'vit_s16_baseline is missing -- resnet18_baseline depends on it.'
print('Ready:', sorted(f for f in os.listdir(REPO_FOLDER) if f.endswith(('.py', '.yaml'))))

## 4. Unzip the split dataset
The same `dataset_split.zip` used by every other experiment. Copied once to the
VM's local SSD and unzipped there, because reading tens of thousands of small
files straight from the Drive FUSE mount is orders of magnitude slower.

In [ ]:
import os, time, shutil, zipfile

DRIVE_ZIP = '/content/drive/MyDrive/Dissertation_Thesis/dataset_split.zip'
LOCAL_ZIP = '/content/dataset_split.zip'
DATA_ROOT = '/content/dataset_split'

if not os.path.isdir(os.path.join(DATA_ROOT, 'train')):
    if not os.path.exists(LOCAL_ZIP):
        t0 = time.time(); shutil.copy(DRIVE_ZIP, LOCAL_ZIP)
        print(f'Copied zip Drive->local in {time.time()-t0:.0f}s')
    t1 = time.time()
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        for info in z.infolist():
            name = info.filename.replace('\\', '/')   # Windows backslash -> /
            if name.endswith('/'):
                continue
            target = os.path.join('/content', name)
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(info) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)
    print(f'Extracted (backslash-safe) in {time.time()-t1:.0f}s')
else:
    print(f'{DATA_ROOT} already present, skipping.')

for sub in ['train', 'val', 'eval']:
    p = os.path.join(DATA_ROOT, sub)
    counts = {c: len(os.listdir(os.path.join(p, c))) for c in sorted(os.listdir(p))}
    print(f'{sub:6s}:', counts)

## 5. Run the full comparison (one cell, both variants)

Writes the Colab-specific paths into the config, then runs `compare_resnet18.py`,
which trains both variants back to back, evaluates each on the held-out test set
(saving embeddings for later analysis) and prints the summary.

Roughly **1 to 1.5 hours per variant** on a T4, so about **2 to 3 hours total**.
Logs stream one clean line per epoch.

Safe to interrupt: results are appended to `resnet18_comparison_results.csv`
after each variant, and re-running this cell skips whatever already finished.

In [ ]:
import os, sys, yaml, subprocess

PROJECT   = '/content/room-classification/resnet18_baseline'
DATA_ROOT = '/content/dataset_split'
RUNS_DIR  = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
if os.path.isdir(PROJECT):
    os.chdir(PROJECT)

cfg = yaml.safe_load(open('config_resnet18.yaml'))
cfg['data']['train_dir'] = DATA_ROOT + '/train'
cfg['data']['val_dir']   = DATA_ROOT + '/val'
cfg['data']['eval_dir']  = DATA_ROOT + '/eval'
cfg['paths']['output_root']    = RUNS_DIR
cfg['training']['num_workers'] = os.cpu_count()
yaml.safe_dump(cfg, open('config_resnet18_colab.yaml', 'w'), sort_keys=False)
print('config written:',
      cfg['training']['epochs'], 'epochs, batch', cfg['training']['batch_size'],
      '| lambda =', cfg['training']['center_loss_weight'])

env = {**os.environ, 'TQDM_DISABLE': '1', 'PYTHONUNBUFFERED': '1'}
p = subprocess.Popen([sys.executable, '-u', 'compare_resnet18.py',
                      '--config', 'config_resnet18_colab.yaml'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
for line in p.stdout:
    sys.stdout.write(line.decode('utf-8', 'replace')); sys.stdout.flush()
print('exit code:', p.wait())

## 6. (Optional) Embedding geometry of the ResNet runs

Runs the same compactness analysis used for the ViT, on the embeddings saved in
step 5. This answers whether Center Loss reshapes the ResNet feature space the
same way it reshaped the ViT one: clusters tighten, but inter-class distances
shrink just as much, so separability does not improve.

In [ ]:
import glob, os, sys, subprocess

RUNS_DIR = '/content/drive/MyDrive/Dissertation_Thesis/dissertation_runs'
ce  = sorted(glob.glob(f'{RUNS_DIR}/*_resnet18_crossentropy'))
cl  = sorted(glob.glob(f'{RUNS_DIR}/*_resnet18_crossentropy_centerloss'))
assert ce and cl, 'run cell 5 first'

env = {**os.environ, 'TQDM_DISABLE': '1', 'PYTHONUNBUFFERED': '1'}
p = subprocess.Popen([sys.executable, '-u',
                      '../vit_s16_baseline/analyze_embeddings.py',
                      '--runs', ce[-1], cl[-1],
                      '--labels', 'ResNet-18 (CE)', 'ResNet-18 (CE + Center Loss)',
                      '--out-dir', '../figuri'],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
for line in p.stdout:
    sys.stdout.write(line.decode('utf-8', 'replace')); sys.stdout.flush()
p.wait()